# Qwen2.5-7B on GPURig2: why does its capture drop a card?

A copy of `persona-lsh.ipynb` cut down to one model in two dtypes, with a hardware test before capture.
Qwen2.5-7B's capture has dropped a riser card (Xid 79) six times on GPURig2 and no other model's has (about 3%
of its batch starts, against none in about 2,100). Every drop came in the first 0.6 s of a new forward pass,
five of six at a batched start, and its batched pass is slower than larger models' (3.0 s against 2.0-2.3 s for
the same prompts). agent-harness's `ops/gpurig2/history.md` ("The Qwen2.5-7B drops, reread") has the evidence.

**What differs from `persona-lsh.ipynb`:**

- **Two runs of one model:** `qwen2.5-bf16`, then `qwen2.5-fp16` (`ONLY`). A run's `dtype` is fixed, with no
  fallback, and each run has its own load test. bf16 goes first: the same weights through other kernels.
- **Section 3a, after the load tests and before capture:** for each run and each card order in `CARD_ORDERS`,
  the model is reloaded so that another card holds the first layers (card 2, then 1, then 0). Each load times
  the batched pass against single rows, splits a pass into seconds per card, runs the torch profiler, and then
  makes `PATH_STARTS` batched starts, the step that has dropped cards.
- **Outputs** go to `~/persona-lsh-output/qwen2_5-lsh/<experiment>/`, apart from the real runs. The draw is the
  same seed as randX, so the prompts are the ones that dropped cards. 3a's numbers are in `batched_path.json`.
- **Telemetry** goes to `~/gpu-telemetry/qwen2_5-lsh_<date_time>/`.
- `CHECK_JUDGE` is off (the judges were checked in `persona-lsh.ipynb`), and the pair compared in section 7 is
  bf16 against fp16.

**A dropped card stops the kernel's use of it until a reboot.** After the reboot, run all again: 3a skips every
test that finished, failed or never ended, so the next card order runs. `batched_path.json` says which was which.

Not run on the cards yet: 3a was written on the laptop and its functions checked on a CPU with a small model.

The rest of this page is `persona-lsh.ipynb`'s.

## persona-lsh on GPURig: the first-round models

Runs this repo's expanded experiment (`demo/expanded_collect.py` + `demo/expanded_analyze.py`) on the rig's three
GTX 1080 Ti cards for every first-round model that agent-harness's `ops/gpurig/pull-hf-models.py` has downloaded.

**What comes from [agent-harness](https://github.com/jo5iah/agent-harness)** (installed editable in the `gpu`
env; the notebook finds the clone through `harness.__file__`): the judge (`harness.load_model_profile`,
`harness.structured`), the rig's config (`configs/rig.yaml`), the telemetry and download scripts
(`ops/gpurig/`), and the model list and its reasons (`lsh-models.md`). The rig's setup and history are in its
`ops/gpurig2/`. Paths such as `ops/...` and `lsh-models.md` below are in that repo.

It goes further than
the published runs in the directions the notes ask for:

- **More models and pairs** (`lsh-models.md`; the NSF plan's "second model family"). Each pair shares an
  architecture, so their persona vectors can be compared directly: Qwen2.5-7B ↔ OpenReasoning-Nemotron-7B,
  Llama-3.1-8B ↔ Nemotron-Nano-8B, Qwen3-8B with thinking off ↔ on. gemma-4-12B ↔ Bolt-E4B differ in size,
  so only their layer profiles are compared.
- **Local judges.** gemma-qat on `ollama@small` sets the gate and gpt-oss on `ollama@big` is the second opinion
  (since 2026-11-19; three judges by consensus on 2026-11-18, gpt-oss alone before),
  through `harness.load_model_profile` and `harness.structured`, replace persona-lsh's OpenAI `gpt-5.4` judge. Section 4 first scores the published Qwen2.5 and Llama
  responses and compares gpt-oss's verdicts with gpt-5.4's (README direction 6: the judge).
- **A layer-by-layer sweep** of single-layer accuracy (README direction 2), cosine against RP-LSH.
- **More traits** if `TRAITS` lists them (README direction 3; the NSF plan's "1-2 more traits"). The default is
  the published three, so the results compare with the README.
- **Persona-vector transfer within a pair:** does a persona digest built on one model classify the other's
  activations? That's the "standardizable" claim in `lsh_proposal.md`, tested between fine-tunes of one
  architecture.

**Downloaded isn't the same as testable.** Before any capture, each model has to pass the
`gpu_capabilities.ipynb` lessons twice: from its config alone (1c: architecture, attention-only, no quantized
weights, fits the measured free memory in fp16) and by loading it on the cards (2b: nothing on the CPU, no
missing weights, every layer's hidden states present, non-zero and finite in fp16 or bf16, a readable reply).
Only models that pass both are captured, and the summary lists the rest with the reason.

**How the runs are set up** (from `gpu_capabilities.ipynb`'s lessons learned and `lsh-models.md`):

- Hidden states need transformers, so every model loads from Hugging Face safetensors in the local cache
  (`HF_HUB_OFFLINE=1`), split over the cards with `device_map="auto"`. Splitting is almost free since
  `ForceP2P=0`. Section 1b checks that copies between cards still arrive intact.
- **fp16, with bf16 as the fallback.** Each model is probed in fp16 first. If any hidden state is not finite,
  it is reloaded in bf16 (emulated on Pascal, about 24% slower, but it can't overflow).
- **Ollama is unloaded first** (`MAKE_ROOM`), and the judge runs only after capture: gpt-oss needs cards 0 and 1.
- **Multimodal wrappers** (gemma-4-12B-it, Bolt-E4B) keep their layer count in `text_config`. The notebook
  counts layers from the hidden states it gets back, not from `config.num_hidden_layers`.

**Differences from persona-lsh's scripts**, all deliberate:

- Hidden states are read on the generated token ids themselves, up to the first end-of-turn token. persona-lsh
  decodes the reply and tokenizes prompt + text again, which can shift token boundaries.
- Generation is batched (`GEN_BATCH`, left padding). Greedy output can differ slightly from unbatched runs
  in fp16. Set `GEN_BATCH = 1` to match persona-lsh exactly.
- Thinking runs (Qwen3 thinking on, OpenReasoning) get `THINK_NEW_TOKENS` instead of 120. Their persona
  vector is averaged over the whole reply (`all_<trait>_diffs.pt`, as persona-lsh does) and also over the
  answer after `</think>` alone (`all_<trait>_answer_diffs.pt`). The judge sees only the answer.

Which questions and framings each run gets is set by `EXPERIMENT` (section 3). Outputs go to `OUT_DIR/<run>/` in persona-lsh's layout (`all_<trait>_diffs.pt`, `responses.json`,
`judge_scores.json`, `eliciting_questions.json`), so `demo/expanded_analyze.py --data_dir` also works on them.
Each run resumes where it stopped: a finished trait or judged run is not redone.

### Environment (on the rig): the `gpu` kernel

Torch runs on the cards, so this uses the `gpu` env. The judge goes through `harness`, so the `gpu` env needs
harness too (editable, so it follows that repo):

```bash
conda activate gpu
pip install -e ~/agent-harness          # harness + langchain-ollama, pandas, pydantic for the judge
pip install -U transformers       # gemma-4 and Bolt-E4B need a recent one (pull-hf-models.py --list checks)
```

Run it headless in `tmux` (capture for 9 runs takes hours) and keep the outputs as the record:

```bash
cd ~/persona-lsh && jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 experiments/qwen2_5-lsh.ipynb
```

**Restart the kernel before running.** The next cell sets `CUDA_DEVICE_ORDER` and `HF_HUB_OFFLINE`, which
only take effect before torch and huggingface_hub are imported.

In [ ]:
import os

# Card numbers in PCI bus order, as in nvidia-smi and the Ollama units. Load only from the local cache:
# huggingface_hub's own downloads hung on the rig's link (lsh-models.md). Both before any import below.
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import gc
import importlib.util
import json
import subprocess
import time
import urllib.request
from collections import Counter
from pathlib import Path

import yaml

FINDINGS = {}  # question -> answer; summarized at the end


def note(key, value):
    FINDINGS[key] = value
    print(f"{key}: {value}")

## Parameters

`RUNS` names each run and the Hugging Face repo it uses. A repo that isn't fully downloaded is skipped
(section 1 says which). Qwen3-8B runs twice, with thinking off and on. Per-run options:

- `template`: keyword arguments for the chat template (Qwen3's `enable_thinking`).
- `system_prefix`: put before the persona instruction in the system prompt. Nemotron-Nano-8B switches its
  reasoning with a `detailed thinking on/off` system prompt, so it gets `detailed thinking off`, to pair
  with Llama-3.1-8B's plain replies.
- `think`: the model writes a `<think>` block first. It gets `THINK_NEW_TOKENS`, and the judge reads the
  answer after `</think>`.

Not verified before the first rig run: whether granite-4.2, gemma-4 and Bolt-E4B think by default, and with
which markers. Section 1 prints each run's rendered prompt, so check it there.

In [ ]:
# This repo (persona-lsh): its root holds demo/ and lsh/, and this notebook is in experiments/. Found from the
# working directory, then at ~/persona-lsh (the rig) or one level down (the dev machine). PERSONA_LSH overrides.
_where = ([Path(os.environ["PERSONA_LSH"])] if "PERSONA_LSH" in os.environ
          else [Path.cwd(), *Path.cwd().parents, Path.home() / "persona-lsh", Path.home() / "persona-lsh" / "persona_vectors"])
PERSONA_LSH = next((p for p in _where if (p / "demo" / "expanded_analyze.py").is_file()), None)
assert PERSONA_LSH, "no persona-lsh clone (with demo/expanded_analyze.py) above this notebook or at ~/persona-lsh; set PERSONA_LSH"
OUT_ROOT = Path.home() / "persona-lsh-output" / "qwen2_5-lsh"   # outside both repos, and apart from the real runs

# Which questions and framings each run gets. The experiment and its seed are picked in the next cell.
QUESTION_SPLIT = {"test": 20, "eval": 20}   # per trait, drawn at random from the extract and eval sets (40)
FRAMING = "random"        # "random": one of the trait's 5 instruction pairs per question; an int: that pair always
DRAW_BY = "experiment"    # "experiment": every run gets the same draw, so models compare on the same questions;
                          # "run": each run draws its own (more variety, noisier comparisons between models)

# The published three. persona-lsh ships seven: add "apathetic", "humorous", "impolite", "optimistic" for
# README direction 3 (each adds 40 generations per run and 40 judge calls).
TRAITS = ["evil", "hallucinating", "sycophantic", "apathetic", "humorous", "impolite", "optimistic"]
N_QUESTIONS = 20          # per trait in the published design (EXPERIMENT = ""); QUESTION_SPLIT otherwise
MAX_NEW_TOKENS = 256      # 120 as published and in rand40, where 75-100% of replies hit it
THINK_NEW_TOKENS = 1024   # runs with think=True: room for the thinking and an answer
GEN_BATCH = 8             # prompts generated together; 1 = exactly persona-lsh's unbatched greedy runs

RUNS = {
    # This notebook's two runs: one model, each dtype fixed (no fallback), each with its own load test.
    "qwen2.5-bf16":     dict(repo="Qwen/Qwen2.5-7B-Instruct", dtype="bfloat16"),
    "qwen2.5-fp16":     dict(repo="Qwen/Qwen2.5-7B-Instruct", dtype="float16"),
    # persona-lsh.ipynb's runs, kept so section 1 accounts for every downloaded model.
    "qwen2.5-7b":       dict(repo="Qwen/Qwen2.5-7B-Instruct"),
    "openreasoning-7b": dict(repo="nvidia/OpenReasoning-Nemotron-7B", think=True),
    "llama3.1-8b":      dict(repo="meta-llama/Llama-3.1-8B-Instruct"),
    "nemotron-nano-8b": dict(repo="nvidia/Llama-3.1-Nemotron-Nano-8B-v1", system_prefix="detailed thinking off"),
    "qwen3-8b":         dict(repo="Qwen/Qwen3-8B", template={"enable_thinking": False}),
    "qwen3-8b-think":   dict(repo="Qwen/Qwen3-8B", template={"enable_thinking": True}, think=True),
    "granite4.2-8b":    dict(repo="ibm-granite/granite-4.2-8b"),
    "gemma4-12b":       dict(repo="google/gemma-4-12B-it"),
    "bolt-e4b":         dict(repo="nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-BF16"),
}
ONLY = ["qwen2.5-bf16", "qwen2.5-fp16"]   # run names to limit this pass to, run in this order

# Same architecture within a pair: persona vectors compare directly, and transfer is tested (section 7).
PAIRS = [("qwen2.5-bf16", "qwen2.5-fp16"), ("qwen2.5-7b", "openreasoning-7b"), ("llama3.1-8b", "nemotron-nano-8b"),
         ("qwen3-8b", "qwen3-8b-think"), ("gemma4-12b", "bolt-e4b")]
# Runs with published persona-lsh results (gpt-5.4 judge, CPU bf16) in PERSONA_LSH/demo/output/expanded/.
PUBLISHED = {"qwen2.5-7b": "Qwen/Qwen2.5-7B-Instruct", "llama3.1-8b": "meta-llama/Llama-3.1-8B-Instruct"}

DTYPE = "float16"        # for runs without a dtype of their own: bf16 is their fallback when fp16 isn't finite
MODEL_CARDS = [0, 1, 2]   # FTW-4 out (ops/gpurig2/setup.md part I): SC-6, SC-7, SC2-A
HEADROOM_GB = 1.0        # left free per card when splitting a model
MAKE_ROOM = True          # unload every Ollama model before capture (big keeps gpt-oss on 0,1)
RESTORE = True            # load them again at the end

TELEMETRY = True         # gpu-telemetry.sh + an fsynced event log for the whole run (section 0)

# Section 3a: the batched path, the profiler and the card order, after the load tests and before capture.
RUN_PATH_TESTS = True
CARD_ORDERS = [[2, 1, 0], [1, 2, 0], [0, 1, 2]]   # tested in this order. order[0] gets the embeddings and first
                          # layers: SC2-A (x16 slot, never dropped), then SC-7, then SC-6 as in every capture
PATH_STARTS = 60          # batched starts (a prefill and one token) per load: at 3% a start, 84% to see a drop
PATH_STEPS = 16           # decode steps timed per load, batched and single
CAPTURE_ORDER = None      # capture's card order: None = device_map="auto" (first layers on MODEL_CARDS[0]),
                          # or an order such as [2, 1, 0] once 3a shows one that holds

RUN_CAPTURE = True
RUN_JUDGE = False        # a hardware test: capture only. Set True to score both runs afterwards
CHECK_JUDGE = False      # the judges were checked against gpt-5.4 in persona-lsh.ipynb
# Judges, in the order they run: model keys in configs/rig.yaml, each with its reasoning setting (None = rig.yaml's).
# Every judge scores every reply (about 4,000 calls each for 7 runs).
JUDGES = {
    "gemma-qat":    False,   # ollama@small, card 2: gemma4 12B qat. Thinking (2,000-3,000 tokens) is far too slow.
                             #   Closest to gpt-5.4 on the published replies (96% same side of 50)
    "gpt-oss":      "low",   # ollama@big, cards 0,1. The judge of 2026-11-15 and rand40: about 1.5 s a call.
                             #   94% with gpt-5.4; scores persona replies 5-16 points low. The second opinion
    # "nemotron-30b": False, # ollama@all. Left out 2026-11-19: 73% with gpt-5.4, and it scores the opposite replies
                             #   about 45 points high on two traits. Its saved scores stay in judges/ and in results 9b-9c
    # A third judge goes here (a rig.yaml key). It is checked against gpt-5.4 first (CHECK_JUDGE).
}
# The gate (judge_scores.json, eliciting_questions.json, sections 5-7): one judge's scores, or "consensus" for the
# median of all JUDGES. Consensus of two is the lower score of each reply, which is nearly gpt-oss's gate.
GATE = "gemma-qat"
POS_THRESHOLD, NEG_THRESHOLD = 50, 50   # persona-lsh's gate: with >= 50 and without < 50
MIN_ELICITING = 4        # a trait with fewer eliciting questions is left out of that run's analysis
                         # (sections 5-7); the run stays in while 2 or more traits are left

CV_FOLDS, CV_SEED = 3, 42
RP_BITS = [256, 1024]
RP_SEED = 42
MULTI_N = 5
assert MIN_ELICITING >= CV_FOLDS, "every kept trait needs a held-out question in each fold"

import harness   # agent-harness, installed editable: the judge, the rig's config, the telemetry scripts
REPO = Path(harness.__file__).resolve().parent.parent   # the agent-harness clone
assert (REPO / "configs" / "rig.yaml").is_file(), f"{REPO} isn't an agent-harness clone (pip install -e it)"
RIG = yaml.safe_load((REPO / "configs" / "rig.yaml").read_text())
OLLAMA_HOST = os.environ.get("HARNESS_RIG_HOST", RIG["host"])
OLLAMA_PORTS = {name: inst["port"] for name, inst in RIG["instances"].items()}
print("persona-lsh:", PERSONA_LSH, "| judges:", JUDGES, "| gate by:", GATE)

## Pick the experiment and its seed

**randX** is a drawn experiment like `rand40` (20 test + 20 eval questions per trait, a random instruction pair
per question, the same draw for every model), named by its design and seed: `rand40-t256-s<seed>` is 40
questions per trait, 256-token replies. **Pick the seed once, before the first capture, and leave it.** A rerun
with the same seed resumes the experiment, since capture and judging skip what's done. A new seed is a new
experiment in its own directory.

`experiment.json` in the experiment's directory records the whole design: seed, split, framing, token budgets,
batch, traits, gate, judges, and the harness and persona-lsh commits. A rerun whose design differs from it stops
before touching anything. `DESIGN = "rand40"` or `""` reopens the 2026-11-16 or 2026-11-15 runs as they ran.


In [ ]:
import secrets

DESIGN = "randX"            # "randX": a new drawn experiment | "rand40": the 2026-11-16 run | "": the published design
SELECTION_SEED = 48213977   # randX only. With the trait name it seeds each trait's draw
print("a fresh seed, if you want another experiment:", secrets.randbelow(10**8))

if DESIGN == "randX":
    EXPERIMENT = f"rand{sum(QUESTION_SPLIT.values())}-t{MAX_NEW_TOKENS}-s{SELECTION_SEED}"
elif DESIGN == "rand40":    # as it ran
    EXPERIMENT, SELECTION_SEED, MAX_NEW_TOKENS, JUDGES, GATE = "rand40", 20261116, 120, {"gpt-oss": "low"}, "gpt-oss"
    QUESTION_SPLIT, FRAMING, DRAW_BY = {"test": 20, "eval": 20}, "random", "experiment"
elif DESIGN == "":          # the first N_QUESTIONS extract questions, every one with instruction pair 0
    EXPERIMENT, MAX_NEW_TOKENS, JUDGES, GATE = "", 120, {"gpt-oss": "low"}, "gpt-oss"
else:
    raise ValueError(f"DESIGN {DESIGN!r}: use 'randX', 'rand40' or ''")
assert GATE == "consensus" or GATE in JUDGES, f"GATE {GATE!r}: use \"consensus\" or one of {list(JUDGES)}"
OUT_DIR = OUT_ROOT / EXPERIMENT if EXPERIMENT else OUT_ROOT
OUT_DIR.mkdir(parents=True, exist_ok=True)


def commit(path):
    r = subprocess.run(["git", "-C", str(path), "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
    dirty = subprocess.run(["git", "-C", str(path), "status", "--porcelain"], capture_output=True, text=True).stdout
    return (r.stdout.strip() + (" (uncommitted changes)" if dirty.strip() else "")) or None


DESIGN_RECORD = {   # what makes two runs comparable; a resumed experiment must match it
    "experiment": EXPERIMENT or "published", "seed": SELECTION_SEED if EXPERIMENT else None,
    "split": QUESTION_SPLIT if EXPERIMENT else {"test": N_QUESTIONS}, "framing": FRAMING if EXPERIMENT else 0,
    "draw by": DRAW_BY if EXPERIMENT else None, "traits": TRAITS, "max new tokens": MAX_NEW_TOKENS,
    "think new tokens": THINK_NEW_TOKENS, "gen batch": GEN_BATCH,
    "gate": {"with at least": POS_THRESHOLD, "without below": NEG_THRESHOLD}}
exp_path = OUT_DIR / "experiment.json"
EXPERIMENT_INFO = json.loads(exp_path.read_text()) if exp_path.exists() else {"created": time.strftime("%F %T")}
changed = {k: {"saved": EXPERIMENT_INFO[k], "now": v} for k, v in DESIGN_RECORD.items()
           if k in EXPERIMENT_INFO and EXPERIMENT_INFO[k] != v}
assert not changed, (f"{OUT_DIR} was started with another design: {changed}. "
                     "Put the parameters back, or pick a new seed for a new experiment.")
EXPERIMENT_INFO |= DESIGN_RECORD | {
    "judges": EXPERIMENT_INFO.get("judges", {}) | JUDGES,   # every judge that has scored this experiment
    "gate by": GATE,                                        # whose scores are the gate as of this pass
    "commits": EXPERIMENT_INFO.get("commits", []) + [{"at": time.strftime("%F %T"), "harness": commit(REPO),
                                                      "persona-lsh": commit(PERSONA_LSH)}]}
exp_path.write_text(json.dumps(EXPERIMENT_INFO, indent=1))
print("outputs:", OUT_DIR, "| design:", {k: v for k, v in DESIGN_RECORD.items() if k != "traits"})


## 0. Telemetry: what was running when a card drops

Card 0 fell off the bus (Xid 79) twice during this notebook's Qwen2.5-7B capture (2026-11-09), but
`gpu_step_load.py --split`, the same model and batching outside Jupyter, ran to completion. So this records
the notebook run itself, in the step tests' format:

- `ops/gpurig/gpu-telemetry.sh` writes per-card watts, temperature, clocks, memory and PCIe link
  (`gpus.csv`), host RAM (`host.csv`), and NVRM/Xid/AER kernel lines (`kernel.log`), once a second.
- `events.csv` gets a start and an end for every step: unloading Ollama, each load test, each model load, each
  trait's generation and hidden-state reading, and each judge pass. `beat` lines record every generated batch and
  every reply read. Each line is fsynced, so it survives a hang.

The last step with a start and no end is where it died. After a reboot, from a shell:

```bash
python ops/gpurig/gpu_step_load.py --summary ~/gpu-telemetry/persona-lsh_<date_time>
tail -5 ~/gpu-telemetry/persona-lsh_<date_time>/events.csv; cat ~/gpu-telemetry/persona-lsh_<date_time>/kernel.log
```

The differences from the passing reproducer are logged too: which processes hold each card (the Ollama services
keep running here; `rig-steppin-2.sh` stops them), what Ollama has loaded, and how much host RAM is left.

In [ ]:
import atexit
import csv
import signal
from contextlib import contextmanager


class EventLog:
    # The events.csv format gpu_step_load.py --summary reads; every line fsynced so a hang keeps it.
    def __init__(self, path, label):
        new = not path.exists()
        self.file = open(path, "a", newline="")
        self.writer = csv.writer(self.file)
        if new:
            self.writer.writerow(["epoch", "card", "step", "event", "value"])
        self.label = label

    def log(self, step, event, value="", show=True):
        self.writer.writerow([f"{time.time():.3f}", self.label, step, event, value])
        self.file.flush()
        os.fsync(self.file.fileno())
        if show:
            print(f"{time.strftime('%H:%M:%S')} [{step}] {event} {value}"[:300], flush=True)


class NoLog:
    def log(self, *args, **kwargs):
        pass


TELEMETRY_DIR = Path.home() / "gpu-telemetry" / time.strftime("qwen2_5-lsh_%F_%H%M%S")
TELEMETRY_PROC = None


def stop_telemetry():
    global TELEMETRY_PROC
    if TELEMETRY_PROC and TELEMETRY_PROC.poll() is None:
        os.killpg(TELEMETRY_PROC.pid, signal.SIGTERM)
    TELEMETRY_PROC = None


if TELEMETRY:
    TELEMETRY_DIR.mkdir(parents=True, exist_ok=True)
    TELEMETRY_PROC = subprocess.Popen([str(REPO / "ops" / "gpurig" / "gpu-telemetry.sh"), str(TELEMETRY_DIR)],
                                      start_new_session=True, stdout=subprocess.DEVNULL)
    atexit.register(stop_telemetry)
    EV = EventLog(TELEMETRY_DIR / "events.csv", "+".join(map(str, MODEL_CARDS)))
    EV.log("setup", "info", f"qwen2_5-lsh.ipynb, ONLY {ONLY}, CARD_ORDERS {CARD_ORDERS}, GEN_BATCH {GEN_BATCH}, "
                            f"telemetry {TELEMETRY_DIR}")
else:
    EV = NoLog()


@contextmanager
def step(name):
    # A start/end (or error) pair in events.csv. Append strings to the yielded list to log them with the end.
    EV.log(name, "start", show=False)
    notes = []
    try:
        yield notes
    except BaseException as exc:
        EV.log(name, "error", f"{type(exc).__name__}: {exc}"[:300])
        raise
    EV.log(name, "end", "; ".join(map(str, notes)), show=False)


def card_processes():
    # "pid name MiB" per card, from nvidia-smi: Ollama's runners and services show up here.
    out = subprocess.run(["nvidia-smi", "--query-compute-apps=gpu_bus_id,pid,process_name,used_memory",
                          "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=30).stdout
    return "; ".join(" ".join(v.strip() for v in line.split(",")) for line in out.strip().splitlines()) or "none"


def host_avail_gib():
    with open("/proc/meminfo") as f:
        return next(int(l.split()[1]) for l in f if l.startswith("MemAvailable")) / 2**20


print("telemetry:", TELEMETRY_DIR if TELEMETRY else "off")

## 1. Which models are downloaded

The first-round list comes from `ops/gpurig/pull-hf-models.py` itself (`MODELS`, without the SmolLM3
fidelity reference). A repo counts as downloaded only if every weight shard in its snapshot is there. For each
one: architecture, layers and hidden size (from `text_config` for multimodal wrappers), and size on disk,
which is about its fp16 size on the cards.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoConfig

spec = importlib.util.spec_from_file_location("pull_hf_models", REPO / "ops" / "gpurig" / "pull-hf-models.py")
pull = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pull)
FIRST_ROUND = [repo for repo, _ in pull.MODELS if repo != "HuggingFaceTB/SmolLM3-3B"]


def on_disk(repo):
    # (snapshot path, status). Complete only if every shard named by the index is in the cache.
    try:
        path = Path(snapshot_download(repo, local_files_only=True))
    except Exception as exc:
        return None, f"not in the cache ({type(exc).__name__})"
    index = path / "model.safetensors.index.json"
    shards = (set(json.loads(index.read_text())["weight_map"].values()) if index.exists()
              else {p.name for p in path.glob("*.safetensors")})
    if not shards:
        return None, "no safetensors"
    missing = [s for s in shards if not (path / s).exists()]  # exists() follows the link to the blob
    if missing:
        return None, f"incomplete: {len(missing)} of {len(shards)} shards missing"
    return path, f"complete, {sum((path / s).stat().st_size for s in shards) / 2**30:.1f} GiB"


PRESENT = {}
for repo in FIRST_ROUND:
    path, status = on_disk(repo)
    info = {"status": status}
    if path:
        try:
            cfg = AutoConfig.from_pretrained(path)
            text = cfg.get_text_config() if hasattr(cfg, "get_text_config") else cfg
            info |= {"path": path, "type": cfg.model_type, "multimodal": hasattr(cfg, "text_config"),
                     "layers": text.num_hidden_layers, "hidden": text.hidden_size,
                     "GiB": float(status.split(", ")[1].split()[0])}
            PRESENT[repo] = info
        except Exception as exc:
            info["status"] += f"; config fails in transformers: {type(exc).__name__}: {str(exc)[:120]}"
    print(f"{repo:44} {info['status']}" + (f"  [{info['type']}, {info['layers']} layers, hidden {info['hidden']}]"
                                            if repo in PRESENT else ""))

note("downloaded", f"{len(PRESENT)} of {len(FIRST_ROUND)} first-round models: {list(PRESENT)}")
note("not fully downloaded", [r for r in FIRST_ROUND if r not in PRESENT])
unlisted = [r for r in PRESENT if not any(x["repo"] == r for x in RUNS.values())]
assert not unlisted, f"downloaded but not in RUNS, add a run for: {unlisted}"
UNTESTABLE = {}   # repo -> why; filled by sections 1c and 2b

## 1b. Make room, and check the cards

As in `gpu_capabilities.ipynb`: unload every model on every instance in `configs/rig.yaml` and wait until no
Ollama runner holds card memory. Then check that a direct copy between each pair of cards arrives intact. A
split model gives gibberish otherwise (the P2P failure fixed on 2026-11-07 by `ForceP2P=0`).

In [ ]:
def ollama_api(port, path, body=None, timeout=60):
    data = None if body is None else json.dumps(body).encode()
    req = urllib.request.Request(f"http://{OLLAMA_HOST}:{port}{path}", data=data,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        raw = r.read().decode("utf-8", errors="replace")
    return json.loads(raw) if raw.strip() else {}


def ollama_loaded():
    out = {}
    for name, port in OLLAMA_PORTS.items():
        try:
            out[name] = [m["name"] for m in ollama_api(port, "/api/ps", timeout=3)["models"]]
        except OSError:
            pass
    return out


def runners_on_cards():
    out = subprocess.run(["nvidia-smi", "--query-compute-apps=process_name", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout
    return "llama-server" in out


UNLOADED = []
if MAKE_ROOM:
    with step("make-room") as notes:
        for inst, models in ollama_loaded().items():
            for model in models:
                ollama_api(OLLAMA_PORTS[inst], "/api/generate", {"model": model, "keep_alive": 0, "stream": False})
                UNLOADED.append((inst, model))
        deadline = time.time() + 120
        while any(ollama_loaded().values()) or runners_on_cards():
            if time.time() > deadline:
                raise TimeoutError(f"still loaded after 120 s: {ollama_loaded()}")
            time.sleep(1)
        notes.append(f"unloaded {[f'{i}/{m}' for i, m in UNLOADED]}")
    note("unloaded from Ollama", [f"{i}/{m}" for i, m in UNLOADED] or "nothing was loaded")
EV.log("make-room", "info", f"processes on cards: {card_processes()}; Ollama loaded: {ollama_loaded()}")

import torch

CARDS = list(range(torch.cuda.device_count()))
kernel_ok = {}
for i in CARDS:   # gpu_capabilities section 2: does this torch build run kernels on sm_61?
    try:
        kernel_ok[i] = torch.ones(1024, device=f"cuda:{i}").mul(2).sum().item() == 2048
    except Exception as exc:
        kernel_ok[i] = f"{type(exc).__name__}: {exc}"
note("kernels run on every card", kernel_ok)
assert all(v is True for v in kernel_ok.values()), "This torch build can't run on the cards (gpu_capabilities section 2)."


def copy_intact(src, dst, mb=64):
    x = torch.randn(mb * 2**18, device=src)
    y = x.to(dst)
    torch.cuda.synchronize(torch.device(src))
    torch.cuda.synchronize(torch.device(dst))
    return torch.equal(y.cpu(), x.cpu())


intact = {f"{i}->{j}": copy_intact(f"cuda:{i}", f"cuda:{j}") for i in MODEL_CARDS for j in MODEL_CARDS if i != j}
note("direct copies intact", intact)
assert all(intact.values()), ("Copies between cards arrive damaged: check /etc/modprobe.d/nvidia-p2p.conf "
                              "(ForceP2P=0) and run ops/gpurig/p2p_check.py.")


def free_gib():
    return {i: torch.cuda.mem_get_info(i)[0] / 2**30 for i in CARDS}


note("torch / CUDA", f"{torch.__version__} / {torch.version.cuda}")
FREE_AT_START = free_gib()
BUDGET_GIB = sum(max(FREE_AT_START[i] - HEADROOM_GB, 0) for i in MODEL_CARDS)
note("free GiB per card", {i: round(v, 2) for i, v in FREE_AT_START.items()})
note("GiB for a model", f"{BUDGET_GIB:.1f} (gpu_capabilities measured 28.8 with Ollama unloaded)")

## 1c. Testable on these cards? Checks from `gpu_capabilities.ipynb`, before loading anything

Being downloaded isn't enough. Each model is checked against what `gpu_capabilities.ipynb` and `lsh-models.md`
found, using only its config and file sizes:

| Check | Why (lesson) |
|---|---|
| transformers knows the architecture, no `auto_map` | no `trust_remote_code`; gemma-4 and Bolt-E4B need a recent transformers |
| attention-only: no Mamba, linear-attention or SSM layers (`layer_types`, `nemotron_h`, ...) | hybrid layers have no fast kernels on sm_61 (Nemotron-H, Qwen3.5) |
| not a quantized checkpoint (`quantization_config`, MXFP4) | gpt-oss's MXFP4 can't run here; persona-lsh needs fp16/bf16 weights |
| fp16 size + `WORK_GIB` fits `BUDGET_GIB` on `MODEL_CARDS` | 28.8 GiB for a model: about 14B in fp16. fp32 doesn't fit 8B models, so fp16 or bf16 only |

Section 2b then loads each model that passes (the load test). Models that fail here are reported with the
reason and skipped.

In [ ]:
import transformers
from transformers import CONFIG_MAPPING

WORK_GIB = 1.5   # beyond the weights: KV cache for GEN_BATCH replies of up to THINK_NEW_TOKENS, and activations
HYBRID_TYPES = {"nemotron_h", "mamba", "mamba2", "jamba", "falcon_h1", "falcon_mamba", "qwen3_next",
                "qwen3_5", "zamba", "zamba2", "bamba", "lfm2", "olmo_hybrid"}


def static_problems(info):
    cfg = json.loads((info["path"] / "config.json").read_text())
    text = cfg.get("text_config") or cfg
    problems = []
    if "auto_map" in cfg:
        problems.append("needs trust_remote_code")
    if cfg.get("model_type") not in CONFIG_MAPPING:
        problems.append(f"{cfg.get('model_type')} unknown to transformers {transformers.__version__}")
    if {cfg.get("model_type"), text.get("model_type")} & HYBRID_TYPES:
        problems.append(f"hybrid architecture ({text.get('model_type') or cfg.get('model_type')})")
    kinds = {str(k).lower() for k in (text.get("layer_types") or [])}
    odd = sorted(k for k in kinds if any(m in k for m in ("mamba", "linear", "ssm", "conv", "recurrent")))
    if odd:
        problems.append(f"non-attention layers: {odd}")
    quant = cfg.get("quantization_config") or text.get("quantization_config")
    if quant:
        problems.append(f"quantized checkpoint ({quant.get('quant_method', '?')})")
    stored = str(cfg.get("dtype") or cfg.get("torch_dtype") or text.get("torch_dtype") or "bfloat16")
    fp16_gib = info["GiB"] * (0.5 if "float32" in stored else 1.0)
    if fp16_gib + WORK_GIB > BUDGET_GIB:
        problems.append(f"too big: {fp16_gib:.1f} GiB in fp16 + {WORK_GIB} > {BUDGET_GIB:.1f} GiB free on cards {MODEL_CARDS}")
    return problems, round(fp16_gib, 1), round(fp16_gib / (BUDGET_GIB / len(MODEL_CARDS)) + 0.5)


for repo, info in PRESENT.items():
    problems, info["fp16 GiB"], cards = static_problems(info)
    if problems:
        UNTESTABLE[repo] = "; ".join(problems)
    print(f"{repo:44} {info['fp16 GiB']:5.1f} GiB fp16, ~{cards} card(s): "
          f"{'FAIL: ' + UNTESTABLE[repo] if problems else 'ok'}")
note("fails the static checks", {r: UNTESTABLE[r] for r in PRESENT if r in UNTESTABLE} or "none")

## 2. Loading, prompting, generating, reading hidden states

persona-lsh's own code is used where it fits: `lsh.RandomProjectionBackend`, `demo/expanded_analyze.py`'s
cross-validation, and `demo/judge.py`'s eval-prompt templates. They load from `PERSONA_LSH` by path (the repo
has two `judge.py` files). Loading and capture are written here, because of the multimodal wrappers, thinking
replies and batching.

A model loads only if it fits on `MODEL_CARDS`: if `device_map="auto"` puts any of it on the CPU or disk, the
load is refused. It is also refused if weights are missing from the checkpoint (a text-only class loaded
from a multimodal checkpoint would leave its layers randomly initialized).

In [ ]:
from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoTokenizer


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


ea = load_module("expanded_analyze", PERSONA_LSH / "demo" / "expanded_analyze.py")  # puts PERSONA_LSH on sys.path
pj = load_module("persona_judge", PERSONA_LSH / "demo" / "judge.py")
from lsh import RandomProjectionBackend

ea.TRAITS = tuple(TRAITS)   # its CV and layer-selection functions read this module global
TRAIT_DATA = {t: json.loads((PERSONA_LSH / "data_generation" / "trait_data_extract" / f"{t}.json").read_text())
              for t in TRAITS}
DTYPES = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}
print("transformers", transformers.__version__)


def max_memory():
    free = free_gib()
    return {i: (f"{max(free[i] - HEADROOM_GB, 0):.1f}GiB" if i in MODEL_CARDS else "0GiB") for i in CARDS}


AUTO_MAP = {}   # repo -> device_map="auto"'s placement on MODEL_CARDS, kept by the first load without an order


def ordered_map(repo, order):
    # The auto placement with its cards renamed: the piece auto put on MODEL_CARDS[k] goes to order[k], so
    # order[0] holds the embeddings and first layers. The pieces keep their sizes; only their cards change.
    # (max_memory can't do this: accelerate sorts its cards.)
    assert sorted(order) == sorted(MODEL_CARDS), f"card order {order}: use each of {MODEL_CARDS} once"
    assert repo in AUTO_MAP, f"{repo}: load it once without an order first (the load test does)"
    return {module: order[MODEL_CARDS.index(card)] for module, card in AUTO_MAP[repo].items()}


def load(run, dtype, order=None):
    # (tokenizer, model, modules per device). Multimodal checkpoints load in their own class first.
    # order: which card gets which piece of the auto placement (ordered_map); None or MODEL_CARDS = auto itself.
    info = PRESENT[run["repo"]]
    auto = order is None or list(order) == MODEL_CARDS
    placement = (dict(device_map="auto", max_memory=max_memory()) if auto
                 else dict(device_map=ordered_map(run["repo"], list(order))))
    tok = AutoTokenizer.from_pretrained(info["path"])
    if tok.chat_template is None:  # some multimodal repos keep the template with the processor
        from transformers import AutoProcessor
        tok.chat_template = AutoProcessor.from_pretrained(info["path"]).chat_template
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    classes = ([AutoModelForImageTextToText, AutoModelForCausalLM] if info["multimodal"]
               else [AutoModelForCausalLM])
    errors = []
    for cls in classes:
        try:
            model, loading = cls.from_pretrained(info["path"], dtype=dtype, output_loading_info=True, **placement)
            break
        except (ValueError, KeyError) as exc:   # this class doesn't take this config
            errors.append(f"{cls.__name__}: {type(exc).__name__}: {str(exc)[:150]}")
    else:
        raise ValueError("; ".join(errors))
    model.eval()
    placed = dict(Counter(str(d) for d in getattr(model, "hf_device_map", {"": "?"}).values()))
    missing = [k for k in loading["missing_keys"]
               if not any(s in k for s in ("vision", "audio", "multi_modal"))]
    problem = (f"weights missing from the checkpoint: {missing[:5]}" if missing else
               f"doesn't fit on cards {MODEL_CARDS}: {placed}" if {"cpu", "disk"} & placed.keys() else None)
    if problem:
        del model
        gc.collect()
        torch.cuda.empty_cache()
        raise MemoryError(problem)
    if auto:
        AUTO_MAP.setdefault(run["repo"], dict(model.hf_device_map))
    return tok, model, placed


def render(tok, run, system, question):
    # The chat prompt as text. Templates without a system role get the instruction in the user turn,
    # as persona-lsh's chat_prompt does (Gemma-2).
    if run.get("system_prefix"):
        system = f"{run['system_prefix']}\n\n{system}"
    kwargs = dict(tokenize=False, add_generation_prompt=True, **run.get("template", {}))
    try:
        return tok.apply_chat_template([{"role": "system", "content": system},
                                        {"role": "user", "content": question}], **kwargs)
    except Exception as exc:
        if "system" not in str(exc).lower():
            raise
        return tok.apply_chat_template([{"role": "user", "content": f"{system}\n\n{question}"}], **kwargs)


def stop_ids(tok, model):
    ids = model.generation_config.eos_token_id
    ids = set(ids if isinstance(ids, list) else [] if ids is None else [ids])
    return ids | ({tok.eos_token_id, tok.pad_token_id} - {None})


def input_device(model):
    return model.get_input_embeddings().weight.device


def generate(tok, model, prompts, max_new):
    # Greedy replies for a batch of prompts: one list of token ids each, cut at the first stop token.
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(input_device(model))
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None,
                             top_k=None, pad_token_id=tok.pad_token_id)
    stops = stop_ids(tok, model)
    replies = []
    for row in out[:, enc["input_ids"].shape[1]:].tolist():
        cut = next((i for i, t in enumerate(row) if t in stops), len(row))
        replies.append(row[:cut])
    return replies


def think_end(tok, gen):
    # Index of the first token after "</think>" in a reply, or None if there is none.
    single = tok.get_vocab().get("</think>")
    if single is not None:
        return gen.index(single) + 1 if single in gen else None
    for k in range(1, len(gen) + 1):   # "</think>" spans several tokens in this vocabulary
        if "</think>" in tok.decode(gen[:k], skip_special_tokens=False):
            return k
    return None


def hidden_means(tok, model, prompt, gen, answer_from=None):
    # {"response": [layers+1, hidden], "answer": ...} means over the reply's tokens, read in one forward pass
    # on prompt + reply ids. An empty span falls back to the last prompt token, as in persona-lsh.
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    ids = torch.tensor([p + gen], device=input_device(model))
    with torch.no_grad():
        hs = model(input_ids=ids, output_hidden_states=True).hidden_states
    spans = {"response": (len(p), len(p) + len(gen))}
    if answer_from is not None:
        spans["answer"] = (len(p) + answer_from, len(p) + len(gen))
    means = {}
    for name, (a, b) in spans.items():
        if b <= a:
            a, b = len(p) - 1, len(p)
        means[name] = torch.stack([h[0, a:b].float().mean(0).cpu() for h in hs])
    finite = all(torch.isfinite(h).all().item() for h in hs)
    return means, finite, max(h.float().abs().max().item() for h in hs)


def free():
    # Callers drop their own references to a model first.
    gc.collect()
    torch.cuda.empty_cache()

### Rendered prompts

The first `with` prompt of each run, without loading weights. Check that the persona instruction lands in the
system turn (or the user turn for templates without one), and how each template starts a reply: an empty
`<think></think>` means thinking is off, an open `<think>` means it's on.

In [ ]:
def candidates():
    # Runs whose model is downloaded and hasn't failed a check yet: those in ONLY, in ONLY's order (which sets
    # the execution order), or every run in RUNS order when ONLY is empty.
    unknown = [n for n in ONLY if n not in RUNS]
    assert not unknown, f"ONLY names runs not in RUNS: {unknown}"
    return {n: RUNS[n] for n in (ONLY or RUNS)
            if RUNS[n]["repo"] in PRESENT and RUNS[n]["repo"] not in UNTESTABLE}


first = TRAIT_DATA[TRAITS[0]]
for name, run in candidates().items():
    try:
        tok = AutoTokenizer.from_pretrained(PRESENT[run["repo"]]["path"])
        if tok.chat_template is None:
            from transformers import AutoProcessor
            tok.chat_template = AutoProcessor.from_pretrained(PRESENT[run["repo"]]["path"]).chat_template
        if tok.chat_template is None:
            raise ValueError("no chat template")
        text = render(tok, run, first["instruction"][0]["pos"], first["questions"][0])
    except Exception as exc:
        UNTESTABLE[run["repo"]] = f"chat template ({name}): {type(exc).__name__}: {str(exc)[:150]}"
        text = f"FAILED: {UNTESTABLE[run['repo']]}"
    print(f"=== {name} ({run['repo']}) ===\n{text}\n")

## 2b. Load test: does each model really run on the cards?

Each model that passed 1c loads once, in `DTYPE`, and must pass every check below before capture uses it.
Each check comes from a failure seen in `gpu_capabilities.ipynb`:

| Check | Why (lesson) |
|---|---|
| every module on `MODEL_CARDS`, none on `cpu`/`disk` | otherwise tok/s and memory aren't GPU numbers (section 7) |
| no weights missing from the checkpoint | a text-only class on a multimodal checkpoint leaves layers random |
| hidden states = layers + 1 (from `text_config`) | multimodal wrappers hide the layer count (lsh-models.md) |
| no layer's hidden states all zero | split models gave all-zero hidden states while P2P was broken |
| all hidden states finite in fp16; otherwise reload in bf16 | fp16 overflow is model-specific; bf16 is the fallback |
| a non-empty greedy reply, printed to read | split models gave gibberish while P2P was broken |

The probe uses the first `with` prompt (the strongest persona instruction) and the run's own template. In this
notebook every run has its own load test, in its own `dtype` with no fallback (two runs share the repo).
`LOAD_TEST` keeps the dtype, speed and placement for sections 3a and 3b.

In [ ]:
def probe(tok, model, run, n_layers):
    data = TRAIT_DATA[TRAITS[0]]
    prompt = render(tok, run, data["instruction"][0]["pos"], data["questions"][0])
    start = time.perf_counter()
    gen = generate(tok, model, [prompt], 48)[0]
    rate = len(gen) / (time.perf_counter() - start)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    with torch.no_grad():
        hs = model(input_ids=torch.tensor([p + gen], device=input_device(model)), output_hidden_states=True).hidden_states
    peaks = [h.float().abs().max().item() for h in hs]
    return {"finite": all(torch.isfinite(h).all().item() for h in hs),
            "hidden states": len(hs), "expected": n_layers + 1,
            "zero layers": [i for i, v in enumerate(peaks) if v == 0],
            "max abs": round(max((v for v in peaks if v == v), default=float("nan")), 1),   # v == v skips NaN
            "tok/s": round(rate, 1), "reply": tok.decode(gen, skip_special_tokens=True).strip()}


LOAD_TEST = {}   # run name -> result (persona-lsh.ipynb keys it by repo; here two runs share one)
FAILED = {}      # run name -> why it failed its load test
for name, run in candidates().items():
    repo = run["repo"]
    info, result, tok, model = PRESENT[repo], {}, None, None
    print(f"\n### {name}: {repo}")
    try:
        dtypes = ([DTYPES[run["dtype"]]] if run.get("dtype")   # a run with its own dtype gets no fallback
                  else [DTYPES[DTYPE]] + ([torch.bfloat16] if DTYPE == "float16" else []))
        for dtype in dtypes:
            start = time.perf_counter()
            with step(f"test {name} {str(dtype)[6:]}") as notes:
                tok, model, placed = load(run, dtype)
                result = {"dtype": str(dtype)[6:], "load s": round(time.perf_counter() - start),
                          "on": placed, **probe(tok, model, run, info["layers"])}
                notes.append({k: v for k, v in result.items() if k != "reply"})
            result["used GiB"] = round(sum(torch.cuda.memory_allocated(i) for i in CARDS) / 2**30, 1)
            del model, tok
            free()
            print(f"  {result['dtype']}: " + ", ".join(f"{k}={v}" for k, v in result.items() if k not in ("reply", "dtype")))
            print(f"  reply: {result['reply'][:200]!r}")
            if result["finite"]:
                break
        problems = [p for p, bad in [
            ("hidden states not finite in fp16 or bf16", not result["finite"]),
            (f"{result['hidden states']} hidden states for {info['layers']} layers", result["hidden states"] != result["expected"]),
            (f"all-zero hidden states in layers {result['zero layers']}", bool(result["zero layers"])),
            ("empty reply", not result["reply"])] if bad]
    except Exception as exc:
        problems = [f"load: {type(exc).__name__}: {str(exc)[:200]}"]
        tok = model = None   # drop the references so free() can release the cards
        free()
    LOAD_TEST[name] = result
    if problems:
        FAILED[name] = "; ".join(problems)
        print("  FAIL:", FAILED[name])

ACTIVE = {n: r for n, r in candidates().items() if n not in FAILED}
TESTABLE = sorted({r["repo"] for r in ACTIVE.values()})
note("load test", {r: {k: v for k, v in t.items() if k != "reply"} for r, t in LOAD_TEST.items()})
note("untestable", (UNTESTABLE | FAILED) or "none")
note("testable models", f"{len(TESTABLE)}: {TESTABLE}")
note("runs this pass", list(ACTIVE))

## 3. Capture

For each testable run: load in the dtype its load test settled on (2b). Then, per trait, generate `with` (pos instruction) and `without` (neg instruction) replies to each
question, read the mean hidden state over each reply at every layer, and save the per-question difference
`with − without` as `[questions, layers+1, hidden]`, persona-lsh's format.

**Which questions, and which framing.** With `EXPERIMENT` set, each run draws its own questions per trait:
the extract and eval sets (20 each, no overlap) are pooled and shuffled, and the first `QUESTION_SPLIT["test"]`
become the **test** set (cross-validation and persona means, as before) and the next `QUESTION_SPLIT["eval"]` the
**eval** set (held out: classified against means built from test only). Each question also gets one of the
trait's 5 instruction pairs at random (`FRAMING = "random"`), used for both its `with` and `without` prompt.
The draw is seeded by `SELECTION_SEED` and the trait (and the run, with `DRAW_BY = "run"`), saved in `selection.json` before the first
capture (so a resumed run asks the same questions), and every row of `responses.json` records its source,
split, framing and the exact prompts sent.

`run_info.json` records the dtype, the load test's results, generation speed, and how many captures weren't
finite (they should be 0: the load test checked only one prompt). A failure stops only that run.

In [ ]:
import random

POOLS = {"extract": TRAIT_DATA,
         "eval": {t: json.loads((PERSONA_LSH / "data_generation" / "trait_data_eval" / f"{t}.json").read_text())
                  for t in TRAITS}}


def draw(name, trait):
    # The questions a run asks for one trait, in capture order: [{source, index, split, framing}, ...].
    if not EXPERIMENT:   # the published design
        return [{"source": "extract", "index": i, "split": "test", "framing": 0} for i in range(N_QUESTIONS)]
    pool = [(s, i) for s in POOLS for i in range(len(POOLS[s][trait]["questions"]))]
    assert sum(QUESTION_SPLIT.values()) <= len(pool), f"{trait}: only {len(pool)} questions"
    rng = random.Random(f"{SELECTION_SEED}/{name}/{trait}" if DRAW_BY == "run" else f"{SELECTION_SEED}/{trait}")
    rng.shuffle(pool)
    n_pairs = len(TRAIT_DATA[trait]["instruction"])
    picks, k = [], 0
    for split, n in QUESTION_SPLIT.items():
        for source, index in pool[k:k + n]:
            picks.append({"source": source, "index": index, "split": split,
                          "framing": rng.randrange(n_pairs) if FRAMING == "random" else int(FRAMING)})
        k += n
    return picks


def selection(name):
    # {trait: picks} for a run: from its selection.json if there is one, else drawn (and saved, with EXPERIMENT).
    path = OUT_DIR / name / "selection.json"
    if path.exists():
        return json.loads(path.read_text())["traits"]
    picks = {t: draw(name, t) for t in TRAITS}
    if EXPERIMENT:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps({"seed": SELECTION_SEED, "draw by": DRAW_BY, "split": QUESTION_SPLIT, "framing": FRAMING,
                                    "traits": picks}, indent=1))
    return picks


def question(trait, pick):
    return POOLS[pick["source"]][trait]["questions"][pick["index"]]


if EXPERIMENT:
    example = draw("example", TRAITS[0])
    print(f"{TRAITS[0]}, a run's draw: " + ", ".join(f"{p['source'][0]}{p['index']}/{p['split']}/f{p['framing']}"
                                                   for p in example[:8]) + ", ...")
    print("framings used:", sorted({p["framing"] for p in example}),
          "| test from extract:", sum(p["source"] == "extract" for p in example if p["split"] == "test"))

### 3a. The batched path, the profiler and the card order

Before capture, for each run (bf16, then fp16) and each order in `CARD_ORDERS` (first layers on card 2, then 1,
then 0): reload the model with that order, then

| Step in `events.csv` | What it does | What it answers |
|---|---|---|
| `timing` | The first capture batch (`GEN_BATCH` prompts of `TRAITS[0]`): each row alone, the batch, and the batch's longest prompt repeated (same shape, no padding). Then seconds per decode step, batched and single | Is the batched pass slower than 8 rows' worth? Is it the padding and mask? Is decode slow too? |
| `cards` | One batched pass and one single row with a synchronize around each piece of the device map | How long each card's burst lasts, and how much time is copies between cards |
| `profile` | torch profiler over a batched pass, a single row, and a batched pass plus 4 steps: operations and kernels by time on the cards, with input shapes | Which operation is the extra second, and whether bf16 runs other kernels |
| `starts` | `PATH_STARTS` batched starts (a prefill and one token), cycling through the trait's batches, a `beat` before each | The step that has dropped cards, at about 3% a start in the captures |

The gentle steps come first, so their numbers are saved before `starts`. `batched_path.json` in the experiment's
directory is rewritten after every step, and each test is marked when it starts.

**If a card drops,** the test's error is saved and the section stops: the card needs a reboot. Run all again
afterwards. A test that is done, failed or started without ending is skipped, so the next order runs. To repeat
a test, delete its entry from `batched_path.json`.

**Reading the result.** A drop that follows the first position (whichever card holds the first layers) is the
load step. A drop that stays with card 0 (SC-6) from any position is that card or its riser. No drop in bf16 and
drops in fp16 is the fp16 kernels. Five of the six capture drops were card 0 in the first position, in fp16.


In [ ]:
from torch.profiler import ProfilerActivity, profile

PATH_FILE = OUT_DIR / "batched_path.json"
PATH = json.loads(PATH_FILE.read_text()) if PATH_FILE.exists() else {}   # "<run> <order>" -> results


def save_path():
    PATH_FILE.write_text(json.dumps(PATH, indent=1))


def sync():
    for c in MODEL_CARDS:
        torch.cuda.synchronize(c)


def timed(fn):
    # Seconds for fn, with every card's queue empty before and after.
    sync()
    start = time.perf_counter()
    fn()
    sync()
    return round(time.perf_counter() - start, 3)


def encode(tok, model, prompts):
    return tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(input_device(model))


def forward(model, enc):
    with torch.no_grad():
        model(**enc)


def new_tokens(tok, model, enc, n):
    # Exactly n new tokens per row, with capture's generate() settings. n = 1 is a prefill and one step.
    with torch.no_grad():
        model.generate(**enc, max_new_tokens=n, min_new_tokens=n, do_sample=False, temperature=None, top_p=None,
                       top_k=None, pad_token_id=tok.pad_token_id)


def card_seconds(model, enc):
    # One forward pass with a synchronize around each piece of the device map (embeddings, each layer, norm,
    # head): seconds per card, including the copy onto it, and what is left ("between": mostly the logits' way
    # back to the first card). The synchronizes make the pass a little slower than an untimed one.
    spent, began, hooks = Counter(), {}, []

    def before(key):
        def hook(module, args):
            sync()
            began[key] = time.perf_counter()
        return hook

    def after(key, card):
        def hook(module, args, output):
            sync()
            spent[card] += time.perf_counter() - began[key]
        return hook

    for key, card in model.hf_device_map.items():
        module = model.get_submodule(key) if key else model
        hooks += [module.register_forward_pre_hook(before(key)), module.register_forward_hook(after(key, card))]
    try:
        total = timed(lambda: forward(model, enc))
    finally:
        for h in hooks:
            h.remove()
    return ({f"card {c}": round(spent[c], 3) for c in sorted(spent, key=str)}
            | {"between": round(total - sum(spent.values()), 3), "total": total})


def profile_rows(fn, top=40):
    # torch profiler over one call: operations and kernels by their own time on the cards, per input shape.
    acts = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if torch.cuda.is_available() else [])
    with profile(activities=acts, record_shapes=True) as prof:
        fn()
        sync()

    def card_us(e):   # named self_device_time_total since torch 2.4, self_cuda_time_total before
        return getattr(e, "self_device_time_total", None) or getattr(e, "self_cuda_time_total", 0) or 0

    rows = [{"op": e.key[:70], "shapes": str(e.input_shapes)[:100], "calls": e.count,
             "card ms": round(card_us(e) / 1e3, 1), "cpu ms": round(e.self_cpu_time_total / 1e3, 1)}
            for e in prof.key_averages(group_by_input_shape=True)]
    rows.sort(key=lambda r: (-r["card ms"], -r["cpu ms"]))
    return rows[:top]


def cards_alive():
    try:
        return all(torch.ones(8, device=f"cuda:{c}").sum().item() == 8 for c in MODEL_CARDS)
    except Exception:
        return False


def path_test(name, run, order):
    key = f"{name} {order}"
    rec = PATH[key] = {"started": time.strftime("%F %T"), "telemetry": str(TELEMETRY_DIR) if TELEMETRY else None}
    save_path()
    tok = model = None
    try:
        with step(f"path {key} load") as notes:
            tok, model, placed = load(run, DTYPES[LOAD_TEST[name]["dtype"]], order)
            rec |= {"dtype": LOAD_TEST[name]["dtype"], "on": placed, "first card": input_device(model).index}
            notes.append(f"{rec['dtype']} on {placed}, first layers on card {rec['first card']}")

        # Capture's own batches for the first trait: the run's draw, each question with then without.
        trait, picks = TRAITS[0], selection(name)[TRAITS[0]]
        instr = TRAIT_DATA[trait]["instruction"]
        jobs = [render(tok, run, instr[p["framing"]][side], question(trait, p))
                for p in picks for side in ("pos", "neg")]
        batches = [jobs[i:i + GEN_BATCH] for i in range(0, len(jobs), GEN_BATCH)]
        lengths = [len(tok(p, add_special_tokens=False)["input_ids"]) for p in batches[0]]
        batch = encode(tok, model, batches[0])
        rows = [encode(tok, model, [p]) for p in batches[0]]
        unpadded = encode(tok, model, [batches[0][lengths.index(max(lengths))]] * len(batches[0]))
        rec["prompt tokens"] = lengths
        forward(model, rows[0])   # warm up: the first call loads the CUDA libraries

        with step(f"path {key} timing") as notes:
            t = rec["seconds"] = {}
            t["single rows, forward"] = [timed(lambda e=e: forward(model, e)) for e in rows]
            EV.log(f"path {key} timing", "beat", "batch, forward", show=False)
            t["batch, forward"] = timed(lambda: forward(model, batch))
            t["batch without padding, forward"] = timed(lambda: forward(model, unpadded))
            t["single row, 1 token"] = timed(lambda: new_tokens(tok, model, rows[0], 1))
            t["batch, 1 token"] = timed(lambda: new_tokens(tok, model, batch, 1))
            t["single row, per step"] = round((timed(lambda: new_tokens(tok, model, rows[0], 1 + PATH_STEPS))
                                               - t["single row, 1 token"]) / PATH_STEPS, 4)
            t["batch, per step"] = round((timed(lambda: new_tokens(tok, model, batch, 1 + PATH_STEPS))
                                          - t["batch, 1 token"]) / PATH_STEPS, 4)
            notes.append(t)
            save_path()

        with step(f"path {key} cards") as notes:
            rec["seconds per card"] = {"batch": card_seconds(model, batch), "single row": card_seconds(model, rows[0])}
            notes.append(rec["seconds per card"])
            save_path()

        with step(f"path {key} profile"):
            try:   # the profiler's field names have changed between torch versions: a failure here isn't a card's
                rec["profile"] = {"batch": profile_rows(lambda: forward(model, batch)),
                                  "single row": profile_rows(lambda: forward(model, rows[0])),
                                  "batch, 1 + 4 tokens": profile_rows(lambda: new_tokens(tok, model, batch, 5))}
            except Exception as exc:
                if not cards_alive():
                    raise
                rec["profile"] = {"error": f"{type(exc).__name__}: {str(exc)[:200]}"}
                print("  profiler failed:", rec["profile"]["error"])
            save_path()

        with step(f"path {key} starts") as notes:
            encs = [encode(tok, model, b) for b in batches]
            starts = rec["batched starts"] = []
            for k in range(PATH_STARTS):
                EV.log(f"path {key} starts", "beat", f"start {k + 1}/{PATH_STARTS}, batch {k % len(encs) + 1}", show=False)
                starts.append(timed(lambda: new_tokens(tok, model, encs[k % len(encs)], 1)))
                if k % 10 == 9:
                    save_path()
            notes.append(f"{len(starts)} starts, median {sorted(starts)[len(starts) // 2]} s")
        rec["done"] = time.strftime("%F %T")
    except BaseException as exc:
        rec["error"] = f"{type(exc).__name__}: {str(exc)[:300]}"
        raise
    finally:
        save_path()
        del model, tok
        free()
    return rec


def path_status(rec):
    return ("done" if "done" in rec else "FAILED: " + rec["error"][:120] if "error" in rec
            else f"started {rec['started']} and never ended (the kernel died: a dropped card hangs it)")


for name, run in (ACTIVE.items() if RUN_PATH_TESTS else []):
    for order in CARD_ORDERS:
        key = f"{name} {order}"
        if key in PATH:
            print(f"{key}: {path_status(PATH[key])}; not run again (delete it from {PATH_FILE.name} to repeat it)")
            continue
        print(f"\n### {key}")
        try:
            rec = path_test(name, run, order)
        except Exception as exc:
            print(f"  FAILED: {type(exc).__name__}: {str(exc)[:300]}")
            if not cards_alive():
                raise RuntimeError(f"a card is gone during {key}: see kernel.log in {TELEMETRY_DIR}. Reboot, then "
                                   "run all again: the tests already tried are skipped.") from exc
            continue
        t = rec["seconds"]
        rows_s = t["single rows, forward"]
        print(f"  first layers on card {rec['first card']}, prompt tokens {rec['prompt tokens']}")
        print(f"  forward: batch {t['batch, forward']} s | the same rows one by one {sum(rows_s):.3f} s "
              f"(each {min(rows_s)}-{max(rows_s)}) | batch without padding {t['batch without padding, forward']} s")
        print(f"  1 token: batch {t['batch, 1 token']} s, single row {t['single row, 1 token']} s | per decode step: "
              f"batch {t['batch, per step']} s, single row {t['single row, per step']} s")
        print(f"  seconds per card, batch: {rec['seconds per card']['batch']}")
        print(f"  seconds per card, single row: {rec['seconds per card']['single row']}")
        for what, rows in rec["profile"].items() if "error" not in rec["profile"] else []:
            print(f"  profile, {what}: the 8 longest on the cards")
            for r in rows[:8]:
                print(f"    {r['card ms']:9.1f} ms  x{r['calls']:<5} {r['op']}  {r['shapes']}")
        starts = sorted(rec["batched starts"])
        print(f"  {len(starts)} batched starts without a drop: {starts[0]}-{starts[-1]} s, median {starts[len(starts) // 2]} s")

note("batched path", {key: path_status(rec) for key, rec in PATH.items()} or "not run")
note("batched path, seconds (batch forward / rows one by one / per batched step)",
     {key: [rec["seconds"]["batch, forward"], round(sum(rec["seconds"]["single rows, forward"]), 3),
            rec["seconds"]["batch, per step"]] for key, rec in PATH.items() if "per step" in str(rec.get("seconds"))})


### 3b. Capture

`qwen2.5-bf16`, then `qwen2.5-fp16`, batched as in `persona-lsh.ipynb`, with the card order `CAPTURE_ORDER`
(`None`: the auto placement, first layers on card 0, as in every capture that dropped).


In [ ]:
def capture(name, run):
    out = OUT_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    todo = [t for t in TRAITS if not (out / f"all_{t}_diffs.pt").exists()]
    if not todo:
        return json.loads((out / "run_info.json").read_text())
    info_path = out / "run_info.json"
    info = json.loads(info_path.read_text()) if info_path.exists() else {}
    max_new = THINK_NEW_TOKENS if run.get("think") else MAX_NEW_TOKENS

    test = LOAD_TEST[name]
    dtype = DTYPES[info.get("dtype", test["dtype"])]   # a resumed run keeps the dtype it started with
    with step(f"{name} load") as notes:
        tok, model, placed = load(run, dtype, CAPTURE_ORDER)
        notes.append(f"{str(dtype)[6:]} on {placed}, card order {CAPTURE_ORDER or 'auto'}, first layers on card "
                     f"{input_device(model).index}, host RAM avail {host_avail_gib():.1f} GiB")
    try:
        info |= {"repo": run["repo"], "dtype": str(dtype)[6:],
                 "load test": {k: v for k, v in test.items() if k != "reply"},
                 "modules per device": placed, "card order": CAPTURE_ORDER or "auto",
                 "first card": input_device(model).index, "max_new_tokens": max_new,
                 "gen batch": GEN_BATCH, "template": run.get("template", {}),
                 "system prefix": run.get("system_prefix"), "transformers": transformers.__version__,
                 "design": {"experiment": EXPERIMENT or "published", "split": QUESTION_SPLIT if EXPERIMENT else
                            {"test": N_QUESTIONS}, "framing": FRAMING if EXPERIMENT else 0, "seed": SELECTION_SEED,
                            "draw by": DRAW_BY}}
        picks_of = selection(name)
        responses_path = out / "responses.json"
        responses = json.loads(responses_path.read_text()) if responses_path.exists() else {}

        for trait in todo:
            picks = picks_of[trait]
            instr = TRAIT_DATA[trait]["instruction"]
            questions = [question(trait, p) for p in picks]
            jobs = [(qi, cond, render(tok, run, instr[p["framing"]][side], questions[qi]))
                    for qi, p in enumerate(picks) for cond, side in (("with", "pos"), ("without", "neg"))]
            EV.log(f"{name} {trait}", "info", f"processes on cards: {card_processes()}; "
                                               f"host RAM avail {host_avail_gib():.1f} GiB")
            replies, start, n_tok = {}, time.perf_counter(), 0
            with step(f"{name} {trait} gen") as notes:
                for i in range(0, len(jobs), GEN_BATCH):
                    batch = jobs[i:i + GEN_BATCH]
                    t0 = time.perf_counter()
                    gens = generate(tok, model, [p for *_, p in batch], max_new)
                    for (qi, cond, prompt), gen in zip(batch, gens):
                        replies[qi, cond] = (prompt, gen)
                        n_tok += len(gen)
                    EV.log(f"{name} {trait} gen", "beat",
                           f"batch {i // GEN_BATCH + 1}/{-(-len(jobs) // GEN_BATCH)}: {sum(map(len, gens))} tokens "
                           f"in {time.perf_counter() - t0:.1f} s, longest {max(map(len, gens))}, GiB per card "
                           f"{[round(torch.cuda.memory_allocated(c) / 2**30, 1) for c in MODEL_CARDS]}")
                notes.append(f"{n_tok} tokens")
            gen_s = time.perf_counter() - start

            rows, diffs, answer_diffs, bad, unfinished = [], [], [], 0, 0
            with step(f"{name} {trait} read"):
                for qi, q in enumerate(questions):
                    row, means = {"q": q, **picks[qi]}, {}   # source, index, split, framing
                    for cond in ("with", "without"):
                        prompt, gen = replies[qi, cond]
                        k = think_end(tok, gen) if run.get("think") else None
                        t0 = time.perf_counter()
                        means[cond], finite, _ = hidden_means(tok, model, prompt, gen, k)
                        EV.log(f"{name} {trait} read", "beat", f"q{qi} {cond}: {len(gen)} reply tokens, "
                                                             f"{time.perf_counter() - t0:.2f} s", show=False)
                        bad += not finite
                        row[cond] = tok.decode(gen, skip_special_tokens=True).strip()
                        row[f"{cond}_tokens"] = len(gen)
                        row[f"{cond}_prompt"] = prompt   # exactly what the model got
                        if run.get("think"):
                            row[f"{cond}_answer"] = (tok.decode(gen[k:], skip_special_tokens=True).strip()
                                                     if k is not None else None)
                            unfinished += k is None
                    diffs.append(means["with"]["response"] - means["without"]["response"])
                    if run.get("think"):  # NaN where either reply never finished thinking
                        a, b = means["with"].get("answer"), means["without"].get("answer")
                        answer_diffs.append(a - b if a is not None and b is not None
                                            else torch.full_like(diffs[-1], float("nan")))
                    rows.append(row)

            torch.save(torch.stack(diffs), out / f"all_{trait}_diffs.pt")
            if run.get("think"):
                torch.save(torch.stack(answer_diffs), out / f"all_{trait}_answer_diffs.pt")
            responses[trait] = rows
            responses_path.write_text(json.dumps(responses, indent=2))
            info.setdefault("traits", {})[trait] = {
                "generated tokens": n_tok, "gen tok/s": round(n_tok / gen_s, 1), "seconds": round(time.perf_counter() - start),
                "non-finite captures": bad, **({"replies that never finished thinking": unfinished} if run.get("think") else {})}
            info_path.write_text(json.dumps(info, indent=2))
            print(f"  {trait}: {info['traits'][trait]}")
            ex = rows[0]
            print(f"    with:    {ex['with'][:150]!r}\n    without: {ex['without'][:150]!r}")
        return info
    finally:
        del model, tok
        free()


CAPTURED = {}
for name, run in (ACTIVE.items() if RUN_CAPTURE else []):
    print(f"\n### {name}: {run['repo']}")
    try:
        CAPTURED[name] = capture(name, run)
    except Exception as exc:
        CAPTURED[name] = {"error": f"{type(exc).__name__}: {exc}"}
        print("  FAILED:", CAPTURED[name]["error"])
        free()
note("capture", {n: i.get("error") or {"dtype": i.get("dtype"), "on": i.get("modules per device"),
                                       "gen tok/s": [t["gen tok/s"] for t in i.get("traits", {}).values()]}
                 for n, i in CAPTURED.items()})

## 4. Judges: gemma-qat sets the gate, gpt-oss is the second opinion

Each judge in `JUDGES` scores every reply in turn. `harness.load_model_profile(judge)` starts its Ollama
instance if needed, unloads whatever shares its cards, and loads it with its rig.yaml settings (the judges run
one after another, after capture). The judge prompt is persona-lsh's
own per-trait `eval_prompt`; `harness.structured` turns the reply into `refusal` + `score`, which map back to
persona-lsh's scores (an integer 0-100 or `"REFUSAL"`). A reply that never finished thinking scores
`"NO_ANSWER"`, and a failed call `"ERROR"`. Both fail the gate, like a refusal.

- Each judge's scores go to `<run>/judges/<judge>.json`. A run judged before there were several judges
  (only `judge_scores.json`) has it adopted as gpt-oss's.
- **The gate** (`judge_scores.json`, and `eliciting_questions.json` that sections 5-7 use) is `GATE`'s scores:
  gemma-qat since 2026-11-19. On the 2026-11-18 run it was the closest judge to gpt-5.4 (96% on the same side
  of 50; gpt-oss 94%, nemotron-30b 73%), and the three-judge consensus landed within 5 eliciting pairs of it in
  every run. gpt-oss still scores everything: results 9b-9c show where the two disagree and whether the
  classification changes. nemotron-30b is left out of `JUDGES`; its saved scores stay.
- **`GATE = "consensus"`** is the earlier rule: per reply, the median of the judges' integer scores when most
  judges gave one; otherwise `"REFUSAL"` when most refused; otherwise the first other verdict. With two judges
  the median is the lower score.
- Changing `GATE` or `JUDGES` on a finished experiment rewrites only the gate files and sections 5-7 (no capture,
  no judging of what's already scored). `experiment.json` records it as "gate by".
- **Judge check (`CHECK_JUDGE`):** each judge first scores the published Qwen2.5 and Llama responses, compared
  with gpt-5.4's: gate agreement and mean score difference (`judge-check/<judge>/`, gpt-oss's in `judge-check/`).
- A judge loads only if it has something left to score, so a rerun for sections 5-7 alone never touches Ollama.

`persona-lsh-results.ipynb` section 9 compares each judge with gpt-5.4, 9b the judges with each other, and 9c
the classification under each judge's gate.


In [ ]:
import statistics

from pydantic import BaseModel, Field

from harness import load_model_profile, quiet_tracing, structured


class Verdict(BaseModel):
    refusal: bool = Field(description="true only if the model clearly refused to answer")
    score: int = Field(ge=0, le=100, description="0 = trait not present at all, 100 = trait strongly present")


JSON_NOTE = ("\n\nGive the answer as JSON instead: refusal (true or false) and score (0-100; use 0 when "
             "refusal is true).")


def judge_one(judge, trait, question, answer):
    if answer is None:
        return "NO_ANSWER"
    try:
        v = judge.invoke(pj.format_eval_prompt(trait, question, answer) + JSON_NOTE)
    except Exception as exc:
        return f"ERROR: {type(exc).__name__}"
    return "REFUSAL" if v.refusal else v.score


def judge_rows(judge, trait, rows, think):
    pick = (lambda r, c: r.get(f"{c}_answer")) if think else (lambda r, c: r[c])
    return [{"qid": i, "pos_score": judge_one(judge, trait, r["q"], pick(r, "with")),
             "neg_score": judge_one(judge, trait, r["q"], pick(r, "without"))} for i, r in enumerate(rows)]


def gate(s):
    return (isinstance(s["pos_score"], int) and isinstance(s["neg_score"], int)
            and s["pos_score"] >= POS_THRESHOLD and s["neg_score"] < NEG_THRESHOLD)


def eliciting(scores):
    return {t: [s["qid"] for s in scores[t] if gate(s)] for t in scores}


def read_scores(path):
    return json.loads(path.read_text()) if path.exists() else {}


def judge_path(name, judge):
    return OUT_DIR / name / "judges" / f"{judge}.json"


def check_path(name, judge):
    # gpt-oss's checks predate the other judges and keep their place.
    return OUT_ROOT / "judge-check" / (f"{name}.json" if judge == "gpt-oss" else f"{judge}/{name}.json")


def captured(name):
    return all((OUT_DIR / name / f"all_{t}_diffs.pt").exists() for t in TRAITS)


def judged(name, judge):
    scores = read_scores(judge_path(name, judge))
    return all(t in scores for t in TRAITS)


def adopt_legacy(name):
    # A run judged before there were several judges has only judge_scores.json, from gpt-oss.
    legacy, mine = OUT_DIR / name / "judge_scores.json", judge_path(name, "gpt-oss")
    if legacy.exists() and not mine.parent.exists():
        mine.parent.mkdir(parents=True)
        mine.write_text(legacy.read_text())


def consensus(values):
    # One verdict from the judges' verdicts on one reply.
    ints = [v for v in values if isinstance(v, int)]
    if 2 * len(ints) > len(values):
        return int(statistics.median_low(ints))   # of an even count, the lower middle: the stricter gate
    if 2 * sum(v == "REFUSAL" for v in values) > len(values):
        return "REFUSAL"
    return next(v for v in values if not isinstance(v, int))


for name in ACTIVE:
    adopt_legacy(name)
TODO = {}
for judge in JUDGES:
    runs = [n for n in ACTIVE if captured(n) and not judged(n, judge)]
    checks = [n for n in PUBLISHED if CHECK_JUDGE and not check_path(n, judge).exists()]
    if runs or checks:
        TODO[judge] = (runs, checks)
print("left to judge:", {j: r + [f"check {c}" for c in k] for j, (r, k) in TODO.items()} if TODO else "nothing",
      "" if RUN_JUDGE else "(RUN_JUDGE = False: not judging)")


In [ ]:
MAX_JUDGE_ERRORS = 0.1   # a judge erroring on more of its check replies than this sits out the pass
DROPPED = {}
for judge, (runs, checks) in (TODO.items() if RUN_JUDGE else []):
    quiet_tracing()
    free()
    with step(f"judge-load {judge}"):
        role = load_model_profile(judge)[judge]
    reasoning = JUDGES[judge]
    model = structured(role.chat_model(**({} if reasoning is None else {"reasoning": reasoning})), Verdict)
    note(f"judge {judge}", f"{role.spec} on ollama@{role.instance}, reasoning "
                           f"{role.settings.get('reasoning') if reasoning is None else reasoning}, for {runs + checks}")

    for name in checks:
        src = PERSONA_LSH / "demo" / "output" / "expanded" / PUBLISHED[name].replace("/", "__")
        if not (src / "responses.json").exists():
            print(f"{name}: no published responses in {src}")
            continue
        responses = json.loads((src / "responses.json").read_text())
        start = time.perf_counter()
        with step(f"judge-check {name} by {judge}"):
            mine = {t: judge_rows(model, t, responses[t], False) for t in TRAITS if t in responses}
        path = check_path(name, judge)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(mine, indent=2))
        print(f"{judge} check {name}: {sum(len(v) for v in mine.values()) * 2} replies in {time.perf_counter() - start:.0f} s")

    # Its saved checks (this pass's or an earlier one's) say whether the judge works at all.
    verdicts = [s[k] for n in PUBLISHED if check_path(n, judge).exists()
                for v in json.loads(check_path(n, judge).read_text()).values() for s in v for k in ("pos_score", "neg_score")]
    share = sum(str(x).startswith("ERROR") for x in verdicts) / max(len(verdicts), 1)
    if share > MAX_JUDGE_ERRORS:
        DROPPED[judge] = f"{share:.0%} of {len(verdicts)} check replies were errors"
    if judge in DROPPED:
        print(f"SKIPPING {judge}: {DROPPED[judge]}. Its check file is saved; delete it to try this judge again.")
        del JUDGES[judge]   # the consensus below uses the other judges
        continue

    for name in runs:
        path = judge_path(name, judge)
        path.parent.mkdir(parents=True, exist_ok=True)
        scores = read_scores(path)
        responses = json.loads((OUT_DIR / name / "responses.json").read_text())
        for t in [t for t in TRAITS if t not in scores]:
            start = time.perf_counter()
            with step(f"judge {name} {t} by {judge}"):   # "judge <run> " first: results section 10 keys on it
                scores[t] = judge_rows(model, t, responses[t], ACTIVE[name].get("think", False))
            path.write_text(json.dumps(scores, indent=2))
            errors = sum(str(s[k]).startswith("ERROR") for s in scores[t] for k in ("pos_score", "neg_score"))
            print(f"{judge} {name}/{t}: {time.perf_counter() - start:.0f} s, {len(eliciting({t: scores[t]})[t])} "
                  f"eliciting, {errors} errors")

# Each judge against gpt-5.4 on the published replies.
check = {}
for judge in JUDGES:
    for name, repo in (PUBLISHED.items() if CHECK_JUDGE else []):
        path = check_path(name, judge)
        src = PERSONA_LSH / "demo" / "output" / "expanded" / repo.replace("/", "__")
        if not (path.exists() and (src / "judge_scores.json").exists()):
            continue
        mine, published = json.loads(path.read_text()), json.loads((src / "judge_scores.json").read_text())
        for t in mine:
            pairs = list(zip(mine[t], published[t]))
            diffs = [abs(a[k] - b[k]) for a, b in pairs for k in ("pos_score", "neg_score")
                     if isinstance(a[k], int) and isinstance(b[k], int)]
            check[f"{judge} {name}/{t}"] = {
                "gate agrees": f"{sum(gate(a) == gate(b) for a, b in pairs)}/{len(pairs)}",
                "eliciting judge / gpt-5.4": f"{sum(map(gate, mine[t]))} / {sum(map(gate, published[t]))}",
                "mean |score diff|": round(sum(diffs) / max(len(diffs), 1), 1),
                "refusals judge / gpt-5.4": f"{sum(a[k] == 'REFUSAL' for a in mine[t] for k in ('pos_score', 'neg_score'))}"
                                            f" / {sum(b[k] == 'REFUSAL' for b in published[t] for k in ('pos_score', 'neg_score'))}",
                "errors": sum(str(a[k]).startswith("ERROR") for a in mine[t] for k in ("pos_score", "neg_score"))}
if DROPPED:
    note("judges skipped this pass", DROPPED)
if check:
    for key, v in check.items():
        print(f"{key:40} {v}")
    note("judge checks against published gpt-5.4 scores", check)


In [ ]:
JUDGED, PER_JUDGE = {}, {}
for name, run in ACTIVE.items():
    if not captured(name):
        continue  # capture didn't finish
    per = {j: read_scores(judge_path(name, j)) for j in JUDGES}
    gate_by = [GATE] if GATE in JUDGES else list(JUDGES)   # "consensus", or the gate judge sat this pass out
    missing = [j for j in JUDGES if not all(t in per[j] for t in TRAITS)]
    if missing:
        print(f"{name}: not yet judged by {missing}, so no consensus")
        continue
    scores = {t: [{"qid": rows[0]["qid"], **{k: consensus([r[k] for r in rows]) for k in ("pos_score", "neg_score")}}
                  for rows in zip(*(per[j][t] for j in gate_by))] for t in TRAITS}
    (OUT_DIR / name / "judge_scores.json").write_text(json.dumps(scores, indent=2))
    elic = eliciting(scores)
    (OUT_DIR / name / "eliciting_questions.json").write_text(json.dumps(elic, indent=2))
    JUDGED[name] = elic
    PER_JUDGE[name] = {j: sum(len(v) for v in eliciting(per[j]).values()) for j in JUDGES}
    picks = selection(name)
    per_split = {t: "+".join(f"{sum(picks[t][q]['split'] == s for q in elic[t])}/{n}"
                             for s, n in Counter(p["split"] for p in picks[t]).items()) for t in TRAITS}
    print(f"{name}: eliciting ({'+'.join(Counter(p['split'] for p in picks[TRAITS[0]]))}) "
          f"{', '.join(f'{t} {per_split[t]}' for t in TRAITS)}; per judge {PER_JUDGE[name]}")
note(f"eliciting questions per run (gate by {GATE if GATE in JUDGES else 'consensus of ' + ', '.join(JUDGES)})", {n: {t: len(e[t]) for t in TRAITS} for n, e in JUDGED.items()})
if len(JUDGES) > 1:
    note("eliciting question pairs per judge", PER_JUDGE)


## 5. Classification: cosine against RP-LSH, 3-fold CV

persona-lsh's `expanded_analyze` procedure on each judged run: per trait, the eliciting questions are split
into `CV_FOLDS` folds; the mean persona vector per trait comes from the training folds, and each held-out
question is classified by the nearest persona (highest cosine, or lowest Hamming distance between RP-LSH
digests). Strategies: `single` (layer at 20/28 of depth), `multi` (top `MULTI_N` layers by coherence in the
middle 30-90%, picked per fold), `all`. Thinking runs get a second row on the answer-only vectors.

With a drawn design (`EXPERIMENT`), cross-validation runs on the **test** questions only, and a second
row per strategy is the **held-out** check: persona means (and layer picks) from all of the test set's eliciting
questions classify the eval set's eliciting questions. The published design has no eval set.

Traits are dropped per run, not runs: a trait with fewer than `MIN_ELICITING` eliciting *test* questions in a run is
left out of that run, and its remaining traits are classified among themselves. A run is skipped only if fewer
than 2 traits are left. So chance (1 / traits kept) differs between runs, and the table gives it on every row:
compare a run's accuracy with its own chance, not with another run's. `kept_traits.json` records each run's
kept and dropped traits, and `predictions.json` each held-out question's prediction (at the largest
`RP_BITS`) for the confusion matrices in `persona-lsh-results.ipynb`.

The published rows are from persona-lsh's own
`expanded_analyze_results.json` (gpt-5.4 judge, CPU bf16; one RP bit count per file, 1024 for these runs
per the README).

In [ ]:
import pandas as pd


def load_run(name, kind="diffs"):
    # (diffs, eliciting test qids, eliciting eval qids) per trait. qids index the captured questions.
    out = OUT_DIR / name
    diffs = {t: torch.load(out / f"all_{t}_{kind}.pt") for t in TRAITS}
    elic = json.loads((out / "eliciting_questions.json").read_text())
    picks = selection(name)
    # Answer-only vectors are NaN where a reply never finished thinking: leave those questions out.
    elic = {t: [q for q in elic[t] if torch.isfinite(diffs[t][q]).all()] for t in TRAITS}
    test = {t: [q for q in elic[t] if picks[t][q]["split"] == "test"] for t in TRAITS}
    held = {t: [q for q in elic[t] if picks[t][q]["split"] != "test"] for t in TRAITS}
    return diffs, test, held


def kept_traits(test):
    # The traits a run keeps for sections 5-7: those with at least MIN_ELICITING eliciting test questions.
    return [t for t in TRAITS if len(test[t]) >= MIN_ELICITING]


def single_default(n_layers):
    return max(1, round(n_layers * 20 / 28))


def classify(diffs, folds, traits, strategy, bits, layer=None):
    # persona-lsh's fold classifier over `traits` only. Its functions read the module global ea.TRAITS.
    saved, ea.TRAITS = ea.TRAITS, tuple(traits)
    try:
        n_layers = diffs[traits[0]].shape[1] - 1
        results = [ea.classify_one_fold(f, diffs, strategy, layer or single_default(n_layers), MULTI_N,
                                        n_layers, bits, RP_SEED) for f in folds]
    finally:
        ea.TRAITS = saved
    preds = [{k: r[k] for k in ("true", "qid", "cosine_pred", "rp_pred")} for fold, _ in results for r in fold]
    return ea.aggregate([r for r, _ in results]), [layers for _, layers in results], preds


def cv(diffs, test, traits, strategy, bits, layer=None):
    saved, ea.TRAITS = ea.TRAITS, tuple(traits)
    try:
        folds = ea.make_cv_fold_splits({t: test[t] for t in traits}, CV_FOLDS, CV_SEED)
    finally:
        ea.TRAITS = saved
    return classify(diffs, folds, traits, strategy, bits, layer)


def held_out(diffs, test, held, traits, strategy, bits, layer=None):
    # One fold: train on every eliciting test question, classify every eliciting eval question.
    return classify(diffs, [{t: (test[t], held[t]) for t in traits}], traits, strategy, bits, layer)


VARIANTS = [(n, "diffs") for n in JUDGED] + [(n, "answer_diffs") for n in JUDGED if RUNS[n].get("think")]
KEPT, PREDICTIONS, rows = {}, {}, []
for name, kind in VARIANTS:
    label = name + (" (answer)" if kind == "answer_diffs" else "")
    diffs, test, held = load_run(name, kind)
    picks = selection(name)
    traits = kept_traits(test)
    dropped = {t: len(test[t]) for t in TRAITS if t not in traits}
    KEPT[label] = {"kept": traits, "dropped": dropped,
                   "eliciting": {t: {"test": len(test[t]), "eval": len(held[t])} for t in TRAITS}}
    base = {"run": label, "eliciting test": "/".join(str(len(test[t])) for t in TRAITS),
            "eliciting eval": "/".join(str(len(held[t])) for t in TRAITS) if any(held.values()) else "-",
            "traits": len(traits), "chance": 1 / len(traits) if traits else None,
            "dropped": ", ".join(f"{t} {n}" for t, n in dropped.items()) or "-"}
    if len(traits) < 2:
        rows.append(base | {"strategy": f"skipped: under 2 traits with {MIN_ELICITING}+"})
        continue
    for strategy in ("single", "multi", "all"):
        checks = [("cv", cv, (diffs, test, traits))]
        if any(held[t] for t in traits):
            checks.append(("held-out", held_out, (diffs, test, held, traits)))
        for check, fn, args in checks:
            row = base | {"strategy": strategy, "check": check}
            for bits in sorted(RP_BITS):   # the last, largest, gives the saved predictions
                agg, layers, preds = fn(*args, strategy, bits)
                row["cosine"] = agg["cosine_acc"]
                row[f"rp {bits}"] = agg["rp_acc"]
                row["n"] = agg["n_total"]
            row["layers (fold 0)"] = layers[0] if len(layers[0]) <= 6 else f"{len(layers[0])} layers"
            for p in preds:   # which split and framing each prediction's question had
                p |= {k: picks[p["true"]][p["qid"]][k] for k in ("split", "framing")}
            PREDICTIONS.setdefault(label, {})[strategy if check == "cv" else f"{strategy} held-out"] = preds
            rows.append(row)

for name, repo in PUBLISHED.items():
    src = PERSONA_LSH / "demo" / "output" / "expanded" / repo.replace("/", "__")
    if (src / "expanded_analyze_results.json").exists():
        pub = json.loads((src / "expanded_analyze_results.json").read_text())
        counts = json.loads((src / "eliciting_questions.json").read_text())
        for strategy, r in pub.items():
            rows.append({"run": f"published {name}", "eliciting test": "/".join(str(len(v)) for v in counts.values()),
                         "traits": len(counts), "chance": 1 / len(counts), "dropped": "-", "strategy": strategy,
                         "check": "cv", "cosine": r["aggregate"]["cosine_acc"], "rp 1024": r["aggregate"]["rp_acc"],
                         "n": r["aggregate"]["n_total"], "layers (fold 0)": r["layers_per_fold"][0]
                         if len(r["layers_per_fold"][0]) <= 6 else f"{len(r['layers_per_fold'][0])} layers"})

ACCURACY = pd.DataFrame(rows)
ACCURACY["n"] = ACCURACY["n"].astype("Int64")
tail = [c for c in ("eliciting test", "eliciting eval", "dropped") if c in ACCURACY]
ACCURACY = ACCURACY[[c for c in ACCURACY if c not in tail] + tail]
pct = lambda v: "" if pd.isna(v) else f"{v:.1%}"
pd.set_option("display.width", 250)
print(f"eliciting: per trait, in order {'/'.join(TRAITS)}\n")
print(ACCURACY.to_string(index=False, na_rep="",
                         formatters={c: pct for c in ["chance", "cosine", *(f"rp {b}" for b in RP_BITS)]}))
ACCURACY.to_csv(OUT_DIR / "accuracy.csv", index=False)
(OUT_DIR / "kept_traits.json").write_text(json.dumps({"min eliciting": MIN_ELICITING, "runs": KEPT}, indent=2))
(OUT_DIR / "predictions.json").write_text(json.dumps({"rp bits": max(RP_BITS), "runs": PREDICTIONS}, indent=2))
note("traits kept per run", {label: f"{len(k['kept'])}: {k['kept']}" for label, k in KEPT.items()})
note("accuracy table", str(OUT_DIR / "accuracy.csv"))

## 6. Layer sweep (README direction 2)

`single` at every layer from 1 to the last, on each run's kept traits (5), cosine and RP-LSH at the largest `RP_BITS`. Depth is relative
(layer / layers), so models of different depth line up. For each run: the best layer, and the band of layers
where cosine accuracy stays at or above 95%. persona-lsh's published picks were 20/28 (Qwen2.5) and 23/32
(Llama), and multi-layer selection kept landing in the upper middle.

In [ ]:
SWEEP, BITS = {}, max(RP_BITS)
for name, kind in VARIANTS:
    label = name + (" (answer)" if kind == "answer_diffs" else "")
    traits = KEPT[label]["kept"]
    if len(traits) < 2:
        continue
    diffs, test, _ = load_run(name, kind)
    n_layers = diffs[traits[0]].shape[1] - 1
    curve = {}
    for layer in range(1, n_layers + 1):
        agg, _, _ = cv(diffs, test, traits, "single", BITS, layer)
        curve[layer] = (agg["cosine_acc"], agg["rp_acc"])
    SWEEP[label] = curve
    band = [l for l, (c, _) in curve.items() if c >= 0.95]
    best = max(curve, key=lambda l: (curve[l][0] + curve[l][1], -abs(l / n_layers - 0.7)))
    print(f"{label:24} {len(traits)} traits (chance {1 / len(traits):.0%}), {n_layers} layers | best {best} "
          f"({best / n_layers:.0%} deep): cosine {curve[best][0]:.0%}, rp{BITS} {curve[best][1]:.0%} | cosine >= 95% at "
          f"{f'{band[0]}-{band[-1]} ({band[0] / n_layers:.0%}-{band[-1] / n_layers:.0%})' if band else 'no layer'}"
          f"{'' if not band or len(band) == band[-1] - band[0] + 1 else ' (with gaps)'}")
    print("    " + " ".join(f"{l}:{c:.0%}/{r:.0%}" for l, (c, r) in curve.items()))

(OUT_DIR / "layer_sweep.json").write_text(json.dumps(
    {label: {"traits": KEPT[label]["kept"], "layers": len(curve), "rp bits": BITS, "curve": curve}
     for label, curve in SWEEP.items()}, indent=2))
note("layer sweep", str(OUT_DIR / "layer_sweep.json"))

## 7. Pairs: do persona vectors carry over between related models?

For each pair in `PAIRS` with the same layer count and hidden size, on the traits both runs keep (5), if
there are at least 2:

- **cos(persona A, persona B)** per trait at A's default single layer: the cosine between the two models'
  mean persona vectors (each over its own eliciting questions). Near 1 means the fine-tune kept the direction.
- **Transfer accuracy:** persona means (and their RP-LSH digests, same projection seed) built on model A
  classify model B's eliciting questions, and B's classify A's. If that holds, one fingerprint library could
  serve a family of fine-tunes.

Pairs that differ in shape (gemma-4-12B ↔ Bolt-E4B) only compare where their sweeps peak, in relative depth.

In [ ]:
def persona_means(diffs, elic, traits, layer):
    return {t: diffs[t][elic[t]][:, layer].mean(0) for t in traits}


def transfer(means, diffs, elic, layer):
    # Accuracy of `means` (one model's personas) on the other model's eliciting questions, over means' traits.
    traits = list(means)
    rp = RandomProjectionBackend(dim=means[traits[0]].numel(), n_bits=BITS, seed=RP_SEED)
    digests = {t: rp.hash_vector(means[t]) for t in traits}
    n = cos_ok = rp_ok = 0
    for t in traits:
        for q in elic[t]:
            v = diffs[t][q, layer]
            cos_ok += max(traits, key=lambda u: ea.cosine_score(v, means[u])) == t
            d = rp.hash_vector(v)
            rp_ok += min(traits, key=lambda u: rp.distance(d, digests[u])) == t
            n += 1
    return {"cosine": round(cos_ok / n, 3), f"rp {BITS}": round(rp_ok / n, 3), "n": n}


PAIR_RESULTS = {}
for a, b in PAIRS:
    if a not in JUDGED or b not in JUDGED:
        continue
    key = f"{a} / {b}"
    shared = [t for t in TRAITS if t in KEPT[a]["kept"] and t in KEPT[b]["kept"]]
    (da, ta, ha), (db, tb, hb) = load_run(a), load_run(b)
    ea_ = {t: ta[t] + ha[t] for t in TRAITS}   # test and eval: every eliciting question
    eb = {t: tb[t] + hb[t] for t in TRAITS}
    if da[TRAITS[0]].shape[1:] != db[TRAITS[0]].shape[1:]:
        peak = {n: max(SWEEP[n], key=lambda l: SWEEP[n][l][0]) / len(SWEEP[n]) for n in (a, b) if n in SWEEP}
        PAIR_RESULTS[key] = {"shapes differ": f"{tuple(da[TRAITS[0]].shape[1:])} vs {tuple(db[TRAITS[0]].shape[1:])}",
                             "best layer, relative depth": {n: f"{v:.0%}" for n, v in peak.items()}}
    elif len(shared) < 2:
        PAIR_RESULTS[key] = {"skipped": f"under 2 traits kept by both (shared: {', '.join(shared) or 'none'})"}
    else:
        layer = single_default(da[TRAITS[0]].shape[1] - 1)
        ma, mb = persona_means(da, ea_, shared, layer), persona_means(db, eb, shared, layer)
        PAIR_RESULTS[key] = {
            "A": a, "B": b, "layer": layer, "traits": shared, "chance": round(1 / len(shared), 3),
            "cos(persona A, persona B)": {t: round(ea.cosine_score(ma[t], mb[t]), 3) for t in shared},
            "A personas -> B": transfer(ma, db, eb, layer),
            "B personas -> A": transfer(mb, da, ea_, layer)}
    print(f"{key}: {PAIR_RESULTS[key]}")
(OUT_DIR / "pairs.json").write_text(json.dumps(PAIR_RESULTS, indent=2))
note("pairs", PAIR_RESULTS)

## Load Ollama's models again

Loads each model section 1b unloaded on its own instance, with the instance's own `keep_alive`. The judge
(gpt-oss on `big`) is likely still loaded already.

In [ ]:
if RESTORE and UNLOADED:
    free()
    for inst, model in UNLOADED:
        try:
            ollama_api(OLLAMA_PORTS[inst], "/api/generate", {"model": model, "stream": False}, timeout=1800)
        except OSError as exc:
            print(f"could not load {inst}/{model}: {type(exc).__name__}: {exc}")
    print("loaded:", ollama_loaded())
else:
    print("Nothing to restore." if RESTORE else "Skipped: RESTORE = False.")

## Summary

In [ ]:
for key, value in FINDINGS.items():
    print(f"- {key}: {value}")

### Telemetry for this run

Stops `gpu-telemetry.sh` and prints `gpu_step_load.py --summary` for the run: per card and per step, the result,
max temperature and watts, min SM clock, max memory, PCIe link and throttle reasons, then the group's total watts
per step and any kernel lines.

In [ ]:
if TELEMETRY:
    EV.log("done", "info", "notebook finished")
    stop_telemetry()
    spec = importlib.util.spec_from_file_location("gpu_step_load", REPO / "ops" / "gpurig" / "gpu_step_load.py")
    gsl = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(gsl)
    gsl.summarize(TELEMETRY_DIR)

## Lessons learned

To fill in from the first full run on the rig, each line citing the section whose numbers back it. Questions
this run should answer:

- Which models needed bf16, and what were their largest fp16 hidden-state values (3)?
- Does gpt-oss agree with gpt-5.4 closely enough on the published responses to stand in as the judge (4)?
- Do the rig's fp16 runs of Qwen2.5-7B and Llama-3.1-8B match the published accuracies and eliciting counts (5)?
- Where in relative depth does each model's persona signal sit, and do the pairs agree (6)?
- Do persona digests transfer between a base model and its NVIDIA or thinking variant (7)?
- How long does capture take per model, and does batching hold up (3)?

The interpretation and graphs of the 2026-11-15 run (8 runs, 7 traits, cards 1-3) are in
`persona-lsh-results.ipynb`, which reads this notebook's files in `OUT_DIR`.